# Regressione logistica: il dataset Default

*Metodi Matematici e Statistici per le Decisioni Economiche · A.A. 2026/27 · Lezione 3*

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vincnardelli/msd/blob/main/codice/3/logistica_default.ipynb)

Il codice della lezione. `default.csv` (dal pacchetto R *ISLR2*) riporta per 10.000 clienti se sono andati in
default (`default`), se sono studenti, il saldo della carta (`balance`) e il reddito (`income`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

DATI = "https://www.vincnardelli.com/msd/dati/"

In [ ]:
Default = pd.read_csv(DATI + "default.csv")
Default.head()

In [ ]:
Default.info()

## Esplorazione

In [ ]:
Default.boxplot(column="balance", by="default")
plt.show()

In [ ]:
colori = Default["default"].map({"No": "tab:blue", "Yes": "tab:orange"})
plt.scatter(Default["balance"], Default["income"], c=colori, s=5)
plt.xlabel("balance")
plt.ylabel("income")
plt.show()

## Modello con `balance`

`smf.logit` vuole una variabile dipendente 0/1: la costruiamo da `default` (Yes = 1).

In [ ]:
Default["y"] = (Default["default"] == "Yes").astype(int)
glm_balance = smf.logit("y ~ balance", data=Default).fit()
glm_balance.summary()

In [ ]:
print(glm_balance.params)              # log-odds
print(np.exp(glm_balance.params))      # odds ratio
print(np.exp(glm_balance.params) - 1)  # variazione percentuale delle odds

## Previsioni per balance = 0, 1000, 2000

In [ ]:
new_data = pd.DataFrame({"balance": [0, 1000, 2000]})
log_odds = glm_balance.predict(new_data, which="linear")
odds = np.exp(log_odds)
print(log_odds)
print(odds)
print(odds / (1 + odds))

In [ ]:
glm_balance.predict(new_data).round(4)   # direttamente la probabilità

## Più variabili

`income` in migliaia di dollari, così il coefficiente si legge meglio.

In [ ]:
Default["income_k"] = Default["income"] / 1000
glm_multi = smf.logit("y ~ balance + income_k + student", data=Default).fit()
glm_multi.summary()

## Confusion matrix e soglie

In [ ]:
def metriche(effettivo, predetto):
    """Confusion matrix (righe = predetto, colonne = effettivo), accuracy, sensitività, specificità."""
    cm = pd.crosstab(predetto, effettivo, rownames=["Predetto"], colnames=["Effettivo"])
    tp = ((predetto == 1) & (effettivo == 1)).sum()
    tn = ((predetto == 0) & (effettivo == 0)).sum()
    fp = ((predetto == 1) & (effettivo == 0)).sum()
    fn = ((predetto == 0) & (effettivo == 1)).sum()
    print(cm, "\n")
    print(f"Accuracy:    {(tp + tn) / len(effettivo):.4f}")
    print(f"Sensitività: {tp / (tp + fn):.4f}")
    print(f"Specificità: {tn / (tn + fp):.4f}")
    return {"TP": tp, "TN": tn, "FP": fp, "FN": fn}

In [ ]:
prob = glm_balance.predict(Default)
for soglia in [0.5, 0.2, 0.8]:
    print(f"--- soglia {soglia}")
    metriche(Default["y"], (prob > soglia).astype(int))